# History, persistence and finite-update geometry

Run cells 1–3 to launch. The background worker stays quiet. Rerun **Status** to see progress; it does not restart the experiment.

Default: SmolLM2 and Pythia-410M Adam; SmolLM2 momentum replication last. Three fresh seeds, identical-state paired branches, frozen forecasts, independently audited curvature. Session budget23 hours; resumable if the full queue takes longer. Read README for the scientific protocol and limits.

In [ ]:
from pathlib import Path
import sys, json, subprocess, importlib
PACKAGE = Path.cwd().resolve()
if not (PACKAGE / "study.py").exists():
    raise FileNotFoundError("Open this notebook in the extracted folder containing study.py.")
sys.path.insert(0, str(PACKAGE))
import study
importlib.reload(study)

# Read-only checks. No automatic package installation or upgrade.
check = """import json, torch, numpy, transformers, datasets, matplotlib
from transformers import LlamaForCausalLM, GPTNeoXForCausalLM
print(json.dumps({'cuda':torch.cuda.is_available(),'torch':torch.__version__,'transformers':transformers.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))
"""
PYTHON = None
errors = []
for candidate in dict.fromkeys([sys.executable, "/home/ubuntu/ml_env/bin/python", "/home/ubuntu/4/env_qwen3/bin/python"]):
    if not Path(candidate).exists():
        continue
    try:
        result = subprocess.run([candidate, "-c", check], capture_output=True, text=True, timeout=120)
        if result.returncode:
            errors.append(candidate + ": " + result.stderr[-1000:]); continue
        info = json.loads(result.stdout.strip().splitlines()[-1])
        if not info["cuda"]:
            errors.append(candidate + ": CUDA unavailable"); continue
        PYTHON = candidate
        print("Ready:", info["gpu"], "| Torch", info["torch"], "| Transformers", info["transformers"])
        break
    except Exception as exc:
        errors.append(candidate + ": " + str(exc))
if PYTHON is None:
    raise RuntimeError("No compatible CUDA environment found. Use your GPU Python; do not replace CUDA Torch.\n" + "\n".join(errors))


## Settings
The existing settings are loaded automatically when resuming. Change scientific settings only before the first launch, or use a new output folder. Operational budgets can be changed on resume.

In [ ]:
OUTPUT = (PACKAGE / "runs" / "history_geometry_followup_v1").resolve()
SETTINGS = json.loads((OUTPUT / "settings.json").read_text()) if (OUTPUT / "settings.json").exists() else study.defaults(str(OUTPUT))
SETTINGS["python"] = PYTHON
SETTINGS["hours"] = 23.0  # cooperative session budget; Launch/resume starts a new budget
# Optional BEFORE first launch: SETTINGS["include_smollm_momentum"] = False
study.validate(SETTINGS)
print("Results:", SETTINGS["output"])
print("Fresh seeds:", SETTINGS["seeds"], "| session budget:", SETTINGS["hours"], "hours")


## Launch / resume
Safe to rerun: an active worker is not duplicated. This continues saved work after a pause.

In [ ]:
launch_result = study.launch(SETTINGS)
print(launch_result.get("status"), "|", SETTINGS["output"])


## Status — rerun this cell
`alive` describes worker liveness, not guaranteed progress. A recent timestamp means an actual reported unit/batch completed. `paused` with a session-budget message means rerun Launch/resume. `failed` includes an error message.

In [ ]:
print(json.dumps(study.status(SETTINGS), indent=2))


## Results — optional, rerun when needed
This prints only completed summaries and an available plot. It does not run training.

In [ ]:
# Uncomment to display the current readable report:
# study.show_results(SETTINGS)


## Stop, restart, export — explicit actions
These are commented so Run All cannot accidentally stop or restart your experiment. Uncomment only the action you need. After Stop, wait for `alive=False`. Restart creates a new folder and preserves the old one. Export produces a ZIP without large model checkpoints.

In [ ]:
# print(study.stop(SETTINGS))
# SETTINGS = study.restart(SETTINGS)  # fresh run in a NEW directory, after worker stops
# print(study.export(SETTINGS))      # upload history_geometry_share.zip for analysis


CPU-only reanalysis is also available via `study.analyze(SETTINGS)` in the notebook's Python if NumPy is installed there; it reads a consistent snapshot and writes `manual_analysis/`. Standard worker reports need no notebook-side scientific dependencies.

Do not change learning rates or fit predictors after inspecting the new test results. Acquisition failures and unresolved numerical audits are outcomes to report.